In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 9 &mdash; Discrete-Time Signals</h1></center>

Modules 7 and 8 treated signals and systems in continuous time and then as vectors and matrices. In practice every geophysical record is a
sequence of numbers: a seismogram sampled every 2 ms, a GPR trace every 0.1 ns, a magnetometer reading every second. This module sets out the
vocabulary of that discrete world: sequences built from unit impulses and steps, random sequences as a model of noise, and five properties
(memory, linearity, time invariance, causality and stability) that decide what a discrete system can and cannot do. It ends with the two
workhorses of digital filtering: the **convolution sum**, the discrete version of Module 7's convolution integral, and **difference equations**,
the discrete cousins of the MATH 225 ODEs, which describe recursive filters with feedback.

# Outline
- [9.1 Continuous, Discrete and Digital Signals](#9.1-Continuous,-Discrete-and-Digital-Signals)
- [9.2 Discrete Signals](#9.2-Discrete-Signals)
- [9.3 Deterministic Sequences](#9.3-Deterministic-Sequences)
  - [9.3.1 Unit impulse](#9.3.1-Unit-impulse)
  - [9.3.2 Unit step](#9.3.2-Unit-step)
- [9.4 Stochastic Signals](#9.4-Stochastic-Signals)
- [9.5 Combining Deterministic Signals and Stochastic Processes](#9.5-Combining-Deterministic-Signals-and-Stochastic-Processes)
- [9.6 Characteristics of Discrete-Time Systems](#9.6-Characteristics-of-Discrete-Time-Systems)
  - [9.6.1 Memory](#9.6.1-Memory)
  - [9.6.2 Linearity](#9.6.2-Linearity)
  - [9.6.3 Time invariance](#9.6.3-Time-invariance)
  - [9.6.4 Causality](#9.6.4-Causality)
  - [9.6.5 Stability](#9.6.5-Stability)
  - [9.6.6 The properties as matrix shapes](#9.6.6-The-properties-as-matrix-shapes)
- [9.7 Discrete Convolution](#9.7-Discrete-Convolution)
- [9.8 Linear Constant-Coefficient Difference Equations](#9.8-Linear-Constant-Coefficient-Difference-Equations)
- [9.9 Summary](#9.9-Summary)
- [Microquestions](#Microquestions)

<a name="9.1"></a>
## 9.1 Continuous, Discrete and Digital Signals

There are three types of signals that are functions of time:
1. **Continuous-time** (analog) - $x(t)$: defined on a continuous range of time *t*, amplitude can be any value.
2. **Discrete-time** - $x[n] = x(nT)$: defined only at discrete instants of time: $t=...-T,0,T,2T...$ where the amplitude can be any value.
3. **Digital** (quantized) - $x_Q[n]$: both time and amplitude are discrete. Signals only defined at $t=...,-T,0,T,2T,...$ and amplitude is confined to a finite set of numbers.

Figure 9.1 shows the three types for one signal.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS

def signal_types_figure():
    """Analog -> sampled -> quantized (Modules 2 and 9 share this code). x(t) = 2cos(2pi(t-1.5)/20), T = 1, Delta = 1."""
    x = lambda t: 2*np.cos(2*np.pi*(t - 1.5)/20)
    T, Delta = 1.0, 1.0
    n = np.arange(-1, 12)
    tt = np.linspace(-1.5, 11.5, 600)
    xn = x(n*T)
    xq = Delta*np.rint(xn/Delta) + 0.0
    fig = plt.figure(figsize=(13, 6.8))
    gs = fig.add_gridspec(2, 3, height_ratios=[0.8, 1])
    # --- top row: the chain x(t) -> sampler -> quantizer -> DSP
    ax0 = fig.add_subplot(gs[0, :]); ax0.axis('off'); ax0.set_xlim(0, 13); ax0.set_ylim(0, 2.2)
    boxes = [(2.2, 'Sampler\n$t = nT$'), (5.6, 'Quantizer\n$Q(x)$'), (9.0, 'DSP\n(computer)')]
    for xc, lab in boxes:
        ax0.add_patch(FancyBboxPatch((xc - 1.0, 0.45), 2.0, 1.3, boxstyle='round,pad=0.05', fc='white', ec=C['time_domain'], lw=2))
        ax0.text(xc, 1.1, lab, ha='center', va='center', fontsize=12)
    # small staircase icon inside the quantizer, drawn from Q itself
    s = np.linspace(-2.2, 2.2, 200); qs = Delta*np.rint(s/Delta)
    ax0.plot(5.6 + 0.62 + 0.12*s, 1.1 + 0.12*qs, color=C['sampled'], lw=1.2)
    arrows = [(0.15, 1.2, '$x(t)$'), (3.2, 4.6, '$x[n]$'), (6.6, 8.0, '$x_Q[n]$'), (10.0, 12.6, 'output')]
    for x0_, x1_, lab in arrows:
        ax0.annotate('', xy=(x1_, 1.1), xytext=(x0_, 1.1), arrowprops=dict(arrowstyle='-|>', lw=1.8, color='k'))
        ax0.text((x0_ + x1_)/2, 1.45, lab, ha='center', va='bottom', fontsize=12)
    # --- bottom row
    titles = ['(a) continuous-time $x(t)$', '(b) discrete-time $x[n] = x(nT)$', '(c) digital $x_Q[n]$ ($\\Delta = 1$)']
    for j in range(3):
        ax = fig.add_subplot(gs[1, j])
        ax.axhline(0, color='k', lw=0.8)
        if j == 0:
            ax.plot(tt, x(tt), color=C['continuous'], lw=2)
        else:
            ax.plot(tt, x(tt), color=C['continuous'], lw=1, alpha=0.35)
            vals = xn if j == 1 else xq
            ml, sl, bl = ax.stem(n*T, vals, basefmt=' ')
            plt.setp(sl, color=C['sampled'], lw=1.5); plt.setp(ml, color=C['sampled'], ms=6)
            if j == 2:
                for lev in (-2, -1, 0, 1, 2):
                    ax.axhline(lev*Delta, ls='--', color=C['neutral'], lw=0.8)
        ax.set_xlim(-1.6, 11.6); ax.set_ylim(-2.4, 2.4)
        ax.set_xlabel('$t$ (s)' if j == 0 else '$n$ (samples, $T = 1$ s)'); ax.set_title(titles[j], fontsize=12)
        if j == 0:
            ax.set_ylabel('amplitude')
    plt.tight_layout()
    return fig

signal_types_figure()
plt.show()

<b>Figure 9.1.</b> From an analog signal to a digital one. Top: the chain from $x(t)$ through a sampler ($t = nT$) and a quantizer $Q$ to the computer. (a) The continuous-time signal $x(t) = 2\cos(2\pi(t - 1.5)/20)$. (b) Its samples $x[n] = x(nT)$ with $T = 1$ s, which can take any value. (c) The quantized samples $x_Q[n]$, restricted to the five levels $-2, \dots, 2$ (dashed) with step $\Delta = 1$; the largest quantization error here is 0.414, below $\Delta/2$. <i>(Python-generated.)</i>

In DSP we deal with $x_Q[n]$ because this corresponds with computer-based processing (which is quantized by definition - e.g. 16-bit vs 32-bit system).  In this course we will assume that **discrete-time signal** is equivalent to a **digital signal** (equivalent to saying that the quantizer has infinite resolution).  Thus, we will commonly write continuous and discrete (and quantized) signals as $x(t)$ and $x[n] = x(nT)$, where parentheses and square brackets denote continuity and discreteness, respectively, and $n$ is the integer sample index.

<a name="9.2"></a>
## 9.2 Discrete Signals

Discrete-time signals are commonplace in the digital world, including the types of geophysical data that you will be using in your career. In fact, the numerical examples in the previous sections were largely based on discrete time series (though this was not greatly emphasized). As you have observed, discrete-time signals can be generated using a number of different approaches:
* Recording the output of a discrete system (e.g., a series of coin flips);
* <a href="https://en.wikipedia.org/wiki/Sampling_(signal_processing)">Sampling</a> continuous-time signals in the real world; or
* Specifying them as arrays in computer software such as Python or Matlab.

Let's now examine the consequences of the signals being discrete!

The discrete time signal $x(nT)$ is equal to continuous signal $x(t)$ only at the sampling interval of $t=nT, n=\{...,-2,-1,0,1,2,...\}$, where $T$ is the **sampling period**.  Thus, $x[n]$ is a **sequence of numbers** $\{...,x[-2],x[-1],x[0],x[1],x[2],...\}$ where $n$ is the temporal index.  To emphasize the discrete nature of the signals, we often use *stem* diagrams that clearly indicate the locations of where samples are located as well as their coefficient magnitude.  For example, see the $\cos$ function plotted in Figure 9.2.

In [ ]:
C = DSP_COLORS
nt = 40
t = np.linspace(0, 2*np.pi, nt)              # 40 samples on [0, 2pi], T = 2pi/39 s
tt = np.linspace(0, 2*np.pi, 500)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(tt, np.cos(tt), color=C['continuous'], lw=1.5, label=r'$x(t) = \cos t$')
ml, sl, bl = ax.stem(t, np.cos(t), basefmt='k-', label=r'$x[n] = \cos(nT)$')
plt.setp(sl, color=C['sampled'], lw=1.2); plt.setp(ml, color=C['sampled'], ms=5)
ax.set_xlim(-0.1, 2*np.pi + 0.1); ax.set_ylim(-1.15, 1.15)
ax.set_xlabel('time (s)'); ax.set_ylabel('amplitude'); ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

<b>Figure 9.2.</b> A discrete sequence shown as a stem plot: 40 samples $x[n] = \cos(nT)$ (blue) of the continuous function $\cos t$ (gray) on $[0, 2\pi]$, with sampling period $T = 2\pi/39$ s. <i>(Python-generated.)</i>

<a name="9.3"></a>
## 9.3 Deterministic Sequences

To write a discrete time series one commonly encounters two important [deterministic](https://en.wikipedia.org/wiki/Deterministic_algorithm) sequences: the **Unit Impulse** and the **Unit Step Function**.

<a name="9.3.1"></a>
### 9.3.1 Unit impulse

The **Unit Impulse** function is defined as

$$ \delta[n] =  
\left\{
\begin{array}{ll} 
1, & n=0 \\ 
0, & n \neq 0 
\end{array}
\right.{}
\tag{1}
$$

The impulse is similar to the continuous-time $\delta$ function; however, it is more straightforward because the unit impulse $\delta[n]$ is well defined for every $n$ while the delta function $\delta(t)$ is less easily defined at $t=0$. The $\delta[n]$ is an important function because it serves as the building block of any discrete-time signal $y[n]$.  For example, we may write $y[n]$ as the following **discrete convolution** between coefficients $x[n]$ and the unit impulse $\delta[n]$:

$$ 
\begin{eqnarray}
y[n] &=& ... + x[-1]\delta[n+1]+x[0]\delta[n]+x[1]\delta[n-1]+... \\
\, & = & \sum_{k=-\infty}^{\infty} x[k] \delta[n-k]
\end{eqnarray}
\tag{2}
$$

We will look at discrete convolution in a section below.

<a name="9.3.2"></a>
### 9.3.2 Unit step

The **Unit Step** function is defined as a step up from zero to one at a particular location defined by $n$. For example, the step up at $n=0$ where all positive values are one is given by:

$$ u[n] =  
\left\{
\begin{array}{ll} 
1, & n\ge 0 \\ 
0, & n \lt 0 
\end{array}
\right.{}
\tag{3}
$$

In addition, the unit step function $u[n]$ can be expressed in terms of an infinite sum of $\delta[n]$ functions at $n\ge 0$:

$$ u[n] = \sum_{k=0}^{\infty} \delta[n-k] \tag{4} $$

Conversely, we can use $u[n]$ to represent $\delta[n]$:

$$ \delta[n] = u[n]-u[n-1] \tag{5}$$

which represents the difference between unit step functions offset by a single interval.

<a name="9.4"></a>
## 9.4 Stochastic Signals

Apart from [deterministic signals](https://en.wikipedia.org/wiki/Deterministic_system) like those above, it is also important that we be able to model [stochastic or random signals](https://en.wikipedia.org/wiki/Stochastic_process).  Generally speaking, these stochastic processes arise because the input depends on an external state (e.g., computer clock time, random number drawn from a probability distribution).  There are many different types of [probability distributions](https://en.wikipedia.org/wiki/Probability_distribution) that we can draw from.  For example, you may have heard of the following:
* [Gaussian (or normal)]( https://en.wikipedia.org/wiki/Normal_distribution )
* <a href="https://en.wikipedia.org/wiki/Uniform_distribution_(continuous)">Uniform</a> 
* [Poisson]( https://en.wikipedia.org/wiki/Poisson_distribution ) 

NumPy has built-in random number generators for the various distributions. The code below creates a generator with a fixed seed, `rng = np.random.default_rng(19680801)`, so the figure is reproducible, and then draws random numbers with
* Gaussian: `rng.normal(size=N)` (mean 0, standard deviation 1; scale and shift it to change them)
* Uniform: `rng.uniform(low, high, N)`.

In [ ]:
C = DSP_COLORS
rng = np.random.default_rng(19680801)        # fixed seed for reproducibility
mu, sigma, nrandom = 0, 15, 5000
xg = mu + sigma*rng.normal(size=nrandom)     # Gaussian: mean mu, standard deviation sigma
xu = rng.uniform(-50, 50, nrandom)           # uniform on [-50, 50)
fig, axs = plt.subplots(2, 2, figsize=(12, 7.5))
for row, (xx, name) in enumerate([(xg, 'Gaussian'), (xu, 'uniform')]):
    axs[row, 0].plot(xx, color=C['noise'], lw=0.6)
    axs[row, 0].set_xlim(0, nrandom); axs[row, 0].set_ylim(-75, 75)
    axs[row, 0].set_xlabel('sample number $n$'); axs[row, 0].set_ylabel('value')
    axs[row, 0].set_title(f'({"ac"[row]}) random sequence ({name})')
    axs[row, 1].hist(xx, 51, color=C['noise'], alpha=0.8)
    axs[row, 1].set_xlim(-60, 60); axs[row, 1].set_ylim(0, 350)
    axs[row, 1].set_xlabel('value'); axs[row, 1].set_ylabel('count per bin')
    axs[row, 1].set_title(f'({"bd"[row]}) histogram of ({"ac"[row]})')
axs[0, 1].text(-55, 300, rf'$\mu = {mu}$, $\sigma = {sigma}$' + f'\nsample std = {xg.std():.2f}', fontsize=11)
plt.tight_layout(); plt.show()

<b>Figure 9.3.</b> Two kinds of stochastic signal, 5000 samples each. (a) A sequence drawn from a Gaussian (normal) distribution with mean $\mu = 0$ and standard deviation $\sigma = 15$ (sample standard deviation 14.88). (b) The histogram of (a), which approaches the bell-shaped Gaussian. (c) A sequence drawn from a uniform distribution on $[-50, 50)$. (d) The histogram of (c), which approaches a flat line. <i>(Python-generated.)</i>

<a name="9.5"></a>
## 9.5 Combining Deterministic Signals and Stochastic Processes

There are many scenarios when one chooses to combine deterministic and stochastic models.  Often this is because we know some underlying deterministic physics (i.e., a **linear** operator) or the calibrated response of an instrument (e.g., a geophone).  However, we usually do not know all of the **non-linear** physics to be able to perfectly describe the system, or we have imperfect instruments or unaccounted for stochastic processes (e.g., ambient seismic noise) that we can only describe within a probabilistic framework.

In Module 7 we discussed how to model a linear time-invariant (LTI) system as a linear combination of deterministic and random signals (i.e., noise). For example, the output from an instrument with random noise is often given by a **discrete convolution** model, the sampled form of Module 7's convolutional model of a seismic trace:

$$ y[n] = \sum_{m=0}^{M} x[m] h[n-m] + N[n] \tag{6} $$

A specific example could be where $x[n]$ is the seismic response of the earth, $h[n]$ is the known **linear** response of your recording system, and $N[n]$ is a noise component representing non-linear or unpredictable phenomena (e.g., instrument drift, thermal fluctuations).  We do not know what these are in advance, but could assume that they are represented by, say, some underlying random Gaussian process.

<a name="9.6"></a>
## 9.6 Characteristics of Discrete-Time Systems

A **discrete-time system** is an operator $\mathcal{T}$ that maps an input sequence $x[n]$ into an output sequence $y[n]$ according to:

$$ y[n] = \mathcal{T}\left\{x[n]\right\} \tag{7}$$

Examples of such a $\mathcal{T}$ can be any types of **filters** (e.g., bandpass, lowpass, highpass, bandreject) that you would apply to discrete input data $x[n]$ (e.g., a seismic trace) to generate filtered output data (e.g., a bandpassed seismic trace).

Discrete-time systems have a number of important properties including: 

* **Memory**
* **Linearity**
* **Time-invariance**
* **Causality**
* **Stability**

<a name="9.6.1"></a>
### 9.6.1 Memory

The first important property of a discrete-time system is **memory**.  Basically, this can be thought of like human memory in that you have memory of previous events in your life from the **past**; however, you do not have memories from the **future**.  If you were so unfortunate to have an accident where you lose your previous memories, then you would be said to be without memory or **memoryless**.  Put mathematically, we can define it the following way:

<div class="alert alert-info" role="alert">
A system is considered to be <strong>memoryless</strong> if $y[n]$ at time $n$ depends <strong>only</strong> on $x[n]$ at time $n$; otherwise the system is said to have <strong>memory</strong>.
</div>

**Solution Approach:** Generally, these can be determined by inspection of the system.  Because the system **output** $y[n]$ is at temporal index $n$, all of the **input** terms must either be a function of time sample $n$ (i.e., not $n+1$ or $n-1$) or a constant. 

**EXAMPLE 1-1:** Is this system memoryless?

$$y[n] = \left(x[n]\right)^2 +2 \tag{8}$$

**A:** This system is **memoryless** because both the input and the output are dependent only on time sample $n$ and a constant value 2.

**EXAMPLE 1-2:** Is this system memoryless?

$$y[n] = x[n] + x[n-2] \tag{9}$$

**A:** This system has **memory** because the output $y[n]$ depends on the input at both sample $n$ and $n-2$.

<a name="9.6.2"></a>
### 9.6.2 Linearity

Another important property that we have looked at previously is **linearity**.  This was important when we discussed LTI systems; however, we repeat this again here for completeness.

<div class="alert alert-info" role="alert">
A system is considered to be <strong>linear</strong> if it obeys the principle of <strong>superposition</strong>.  That is, if 


$$y_1[n] = \mathcal{T}\left\{x_1[n]\right\} \quad \mathrm{and}  \quad y_2[n] = \mathcal{T}\left\{x_2[n]\right\} \tag{10}$$


then a system is <b>linear</b> if the following is true for all $n$:

$$
\begin{eqnarray}\mathcal{T}\left\{a x_1[n]+b x_2[n]\right\} &=& a\mathcal{T}\left\{x_1[n]\right\}
+b \mathcal{T}\left\{x_2[n]\right\} \\
\, &=& a y_1[n]+b y_2[n] \tag{11}
\end{eqnarray}$$

where $a$ and $b$ are real numbers. Note that this test combines two items: <b>additivity</b> (the two terms) and <b>homogeneity</b> (scaling via coefficients).  
</div>

**Solution Approach**: A standard approach to determine linearity of a system is as follows: Let

$$ y_i[n] = \mathcal{T}\left\{x_i[n]\right\}, \quad i=1,2,3  \tag{12} $$

Let's first assume that the cumulative **input** $x_3[n]$ is a linear combination of the other two **input** sequences $x_1[n]$ and $x_2[n]$:

$$ x_3[n] = a x_1[n] + b x_2[n] \tag{13}$$

The system is **linear** if and only if the cumulative **output** $y_3[n]$ is the same linear combination of the other two **output** sequences:

$$y_3[n]=a y_1[n]+b y_2[n] \tag{14}$$ 

If this result does not hold, then the system is considered to be **nonlinear**.

**EXAMPLE 2-1**: Determine whether or not the following system with input $x[n]$ and output $y[n]$ is linear: 

$$ y[n] = \sum_{k=-\infty}^{n} x[k] \tag{15}$$

To address this problem, let's write out our solution approach using three (unspecified) sequences that satisfy equation (15):

$$
\begin{eqnarray} 
ay_1[n] &=& a\sum_{k=-\infty}^{n} x_1[k] \tag{16} \\
by_2[n] &=& b\sum_{k=-\infty}^{n} x_2[k] \tag{17} \\
 y_3[n] &=&  \sum_{k=-\infty}^{n} x_3[k] \tag{18} \\
 x_3[k] &=&  ax_1[k]+bx_2[k] \tag{19} 
\end{eqnarray}
$$

Let's start with equation (18) and then substitute in expression in equation (19)
$$
\begin{eqnarray}
y_3[n] &=& \sum_{k=-\infty}^{n} x_3[k] \tag{20} \\
\, & = & \sum_{k=-\infty}^{n} \left(ax_1[k]+bx_2[k]\right)\tag{21}\\
\, & = & a\sum_{k=-\infty}^{n} x_1[k] + b\sum_{k=-\infty}^{n} x_2[k] \tag{22}  \\
\, & = & ay_1[n]+by_2[n] \tag{23}
\end{eqnarray}
$$

where the final two steps involved replacing terms with equations (16) and (17). Thus, by following the solution approach we have demonstrated that the system is **linear**.

**EXAMPLE 2-2:** Determine whether the following system with input $x[n]$ and output $y[n]$ is linear:  

$$ y[n] = 3x^2[n]+2x[n-3] \tag{24} $$

Your first instinct is that this system is likely **non-linear** because of the square of one of the input sequences.  However, to show it formally, let's use inputs $x_i[n]$ where $i=1,2,3$ to define the system outputs $y_i[n]$ in the following way: 

$$y_i[n]=3x_i^2[n]+2x_i[n-3],  \quad i=1,2,3  \tag{25}$$ 

Again, starting with an output $y_3[n]$ and using a linear combination of input sequences, 

$$x_3[n]=ax_1[n]+bx_2[n] \tag{26}$$ 

allows us to write the following system output by **direct substitution**:

$$
\begin{eqnarray}
y_3[n] &=& 3x_3^2[n]+2x_3[n-3]   \tag{27} \\
\, & = & 3\left(ax_1[n]+bx_2[n]\right)^2+2(ax_1[n-3]+bx_2[n-3]) \tag{28} \\
\, & = & 3\left(a^2x_1^2[n]+b^2x^2_2[n] +2ab x_1[n]x_2[n]\right) + 2(ax_1[n-3]+bx_2[n-3]) \tag{29}
\end{eqnarray}
$$

where to get to equation (28) we have directly substituted equation (26) and then expanded the result (equation (29)).  However, if the system is **linear** then we know that the **output** terms must be linear as well and the following must be true:
$$
\begin{eqnarray}
y_3[n] & = & ay_1[n] + by_2[n]  \tag{30}\\
\, & = & a\left( 3x^2_1[n]+2x_1[n-3] \right) + b\left( 3x^2_2[n]+2x_2[n-3]  \right)  \tag{31}\\
\end{eqnarray}
$$

Thus, because equations (29) and (31) are not equal, the system must be **non-linear**.

<a name="9.6.3"></a>
### 9.6.3 Time invariance

The third important property of a discrete time system is **time invariance**. As suggested by the name, applying a **time invariant** system either yesterday, today or tomorrow will not change actual result. On the other hand, systems that are **time variant** are those where the initiation time absolutely matters! Mathematically, we may describe **time invariance** in the following way:

<div class="alert alert-info" role="alert">
A system is <strong>time invariant</strong> if a shift of input causes a corresponding shift in output.  That is, if $y[n]=\mathcal{T}\left\{x[n]\right\}$, then a system is time invariant if and only if (iff)

$$ y[n-n_0] = \mathcal{T}\left\{x[n-n_0]\right\} \tag{32} $$

for all $n_0$.
</div>

**Solution Approach:** The general approach to check for time invariance is to make a substitution $n\rightarrow n-n_0$ for both the input and output variables.  To be time invariant, the shift must not have affected the overall result.

**EXAMPLE 3-1:** Determine whether the following system with input $x[n]$ and output $y[n]$ is time invariant:

$$ y[n] = \sum_{k=-\infty}^{n} x[k] \tag{33} $$

From equation (33) we know that introducing a time lag $n_0$ will result in the following lagged output $y[n-n_0]$:

$$ y[n-n_0]=\sum_{k=-\infty}^{n-n_0} x[k] \tag{34} $$

which we computed effectively by shifting the index according to $n \rightarrow n-n_0$.  Let's now introduce the same lag to the input $x_1[n]=x[n-n_0]$ and see whether the resulting system output is given by $y_1[n]=y[n-n_0]$.  First, let's start by introducing an equation similar to equation (33), but with subscripts $y_1[n]$ and $x_1[n]$.  We're going to eliminate these by substitutions to work our way back to equation (34).

$$ 
\begin{eqnarray} 
y_1[n] &=& \sum_{k=-\infty}^{n} x_1[k] \tag{35}\\
&=& \sum_{k=-\infty}^{n} x[k-n_0] \tag{36}\\
&=& \sum_{l+n_0=-\infty}^{n} x[l], \quad l=k-n_0 \tag{37} \\
&=& \sum_{l=-\infty}^{n-n_0} x[l] \tag{38} \\
&=& y[n-n_0] \tag{39}
\end{eqnarray}
$$ 

The step to equation (36) involved substituting in the definition of a time-lagged signal.  The step in equation (37) involved making the substitution $l=k-n_0$ in the argument of $x[n]$ as well as $k=l+n_0$ in the sum. The next step involved shifting the index in the sum by $n_0$ terms, which also involved  changing the upper bound to account for the fact that we are summing $n_0$ fewer terms in the series (i.e., when $k$ reaches $n$ we note that $l$ would reach $n-n_0$).  Finally, the last result involved using the identity in equation (34).  Thus, because we have shown that $x[n]\rightarrow x[n-n_0]$ generates $y[n]\rightarrow y[n-n_0]$ we have determined that the system is **time invariant**.

**EXAMPLE 3-2:** Determine whether the following system with input $x[n]$ and output $y[n]$ is time invariant:

$$ y[n]=3x[3n] \tag{40} $$

From the given input-output relationship $y[n-n_0]$ is of the form:

$$ y[n-n_0] = 3x[3(n-n_0)] =3x[3n-3n_0] \tag{41} $$

which we calculated in the same way as equation (34) above by directly substituting $n\rightarrow n-n_0$ into equation (40) and expanding the results.  However, if we shift the input by $n_0$ samples to get $x_1[n]=x[n-n_0]$ and $y_1=y[n-n_0]$ like we did in equations (35)&ndash;(39), then the system output will be:

$$ y_1[n] = 3x_1[3n]  \tag{42} $$

$$ y_1[n] = 3x[3n-n_0] \tag{43} $$

$$ y_1[n] = 3x[3n-n_0] \neq y[n-n_0] \tag{44} $$

Thus, because the expressions in equations (41) and (44) are not equivalent, we can say that the system is **time variant**.

<a name="9.6.4"></a>
### 9.6.4 Causality

Another important property in digital signals as well as physics is **causality**.  From physics, we know that any event happening in the present cannot be affected by an action from the future; however, it can be affected by previous actions.  This is the **causality principle**.  In digital signal processing, though, we often have recorded our signals in advance and are processing later on.  Thus, we can play around with causality when defining our digital systems. Put mathematically, 

<div class="alert alert-info" role="alert">
A system is considered <strong>causal</strong> if output $y[n]$ at time $n$ depends on input $x[n]$ up to the current time $n$ but no <strong>future</strong> value (i.e., no dependence on $n+m$ where $m>0$).  However, if output $y[n]$ depends exclusively on $x[n]$ for $n$ at current and <strong>future</strong> values, then the system is considered <strong>acausal</strong> or <strong>anticausal</strong>. Finally, a system that depends on past, current and future values is considered a <strong>non-causal</strong> system.  
    
For <strong>LTI systems</strong> there is an alternate definition: A LTI system is causal if its impulse response $h[n]$ satisfies:

$$h[n]=0, n<0 \tag{45} $$

</div>

**Solution Approach:** Many of the systems can be examined by inspecting the input terms - especially if they are all in the numerator.  (Systems with input terms in the denominator are a bit more challenging to handle since one may use expansions to create approximations, e.g. , binomial expansions of $\frac{1}{1-x}$.)

**EXAMPLE 4-1:** Are the following systems causal?

$$\begin{eqnarray}
y[n] &=& x[n]+x[n+1] \tag{46} \\
y[n] &=& x[n]+x[n-2] \tag{47} \\
y[n] &=& x[n-1]+x[n]+x[n+1] \tag{48}
\end{eqnarray}
$$

1. The first system is **anticausal** because $y[n]$ is dependent on an $n$ at current and **future** times.
2. The second system is **causal** because $y[n]$ is dependent on an $n$ on current and **previous** values.
3. The third system is **non-causal** because $y[n]$ is dependent on an $n$ at past and future values.

<a name="9.6.5"></a>
### 9.6.5 Stability

The final property that we will consider is **stability**. Mathematically, we can define stability in the following way:

<div class="alert alert-info" role="alert">
A system is considered <strong>stable</strong> in the bounded-input, bounded-output (BIBO) sense if every bounded input sequence (i.e., $|x[n]| \le B_x < \infty$ for all $n$) produces a bounded output sequence (i.e., $|y[n]| \le B_y < \infty$ for all $n$). 
    
For an <strong>LTI system</strong> stability also corresponds to 

$$\sum_{n=-\infty}^{\infty} |h[n]| \lt \infty \tag{49} $$

which states that the impulse response must be absolutely summable. It is not enough for each coefficient $h[n]$ to be finite: the sum of their magnitudes must be finite.
</div>

**Solution Approach:** For LTI systems, it is fairly straightforward to apply the formula in equation (49) to determine stability. (For non-LTI systems this gets quite a bit more involved, and we will not be touching these during this course.)

**EXAMPLE 5-1:** Are the following systems stable?

$$\begin{eqnarray}
y[n] &=& x[n]+x[n+1] \tag{50} \\
y[n] &=& 1/x[n]      \tag{51}
\end{eqnarray}
$$

(1) We identify the following impulse response function $h[n] = \delta[n]+\delta[n+1]$, which follows from the property of discrete convolution that we examined before:

$$ y[n] = \sum_{k=-\infty}^{\infty} x[k] h[n-k] = \sum_{k=-\infty}^{\infty} x[k] \left(\delta[n-k]+\delta[n+1-k]\right) = x[n]+x[n+1] \tag{52} $$

Accordingly, we write:

$$\sum_{n=-\infty}^{\infty} \left|h[n]\right| = \sum_{n=-1}^{0} \left|\delta[n]\right|+\left|\delta[n+1]\right| =2 \lt \infty \tag{53}$$

In the second equality we have enforced the summation from -1 to 0 because these are the only $n$ values that will yield non-zero $\delta[\cdot]$ values.  Note that equation (53) indicates that the system is **stable** as per equation (49).

(2) This system is nonlinear, so it has no impulse response and equation (49) does not apply; we go back to the BIBO definition instead. The input $x[n] = 0$ for all $n$ is bounded ($B_x = 0$), but the output $1/x[n]$ is undefined (unbounded) at every $n$. One bounded input with an unbounded output is enough: the system is **unstable**.

<div class="alert alert-warning">
<b>Common misconception:</b> "A stable physical system always gives a stable digital filter." Stability belongs to the discrete system itself, and a careless discretization of a perfectly stable circuit can be unstable (Microquestion 9.2 shows how).
</div>

<a name="9.6.6"></a>
### 9.6.6 The properties as matrix shapes

Every linear system acting on a finite record of $N$ samples can be written as an $N\times N$ matrix
whose $k$-th column is the response to $\delta[n-k]$. Figure 9.4 builds these matrices for five of the systems above, using $N = 8$.

In [ ]:
C = DSP_COLORS
N = 8
def system_matrix(T):
    """Column k = response of the system T to the unit impulse delta[n - k] (Module 8)."""
    return np.column_stack([T(np.eye(N)[:, k]) for k in range(N)])
def shift(x, d):                              # x[n - d] on a finite record, zeros shifted in
    return np.concatenate([np.zeros(d), x[:N - d]]) if d >= 0 else np.concatenate([x[-d:], np.zeros(-d)])
systems = [('(a) $y = 3x[n]$\nmemoryless: diagonal',              lambda x: 3*x),
           ('(b) $y = x[n] + x[n-2]$\ncausal, memory: lower band',  lambda x: x + shift(x, 2)),
           ('(c) $y = x[n] + x[n+1]$\nanticausal: upper band',      lambda x: x + shift(x, -1)),
           ('(d) $y = \\sum_{k\\leq n} x[k]$\ncausal, unstable: lower triangle', lambda x: np.cumsum(x)),
           ('(e) $y = 3x[3n]$\ntime-variant: not Toeplitz',       lambda x: 3*np.concatenate([x[::3], np.zeros(N - len(x[::3]))]))]
fig, axs = plt.subplots(1, 5, figsize=(15, 4.0))
for ax, (title, T) in zip(axs, systems):
    H = system_matrix(T)
    ax.imshow(H, cmap='Blues', vmin=0, vmax=3)
    for i in range(N):
        for j in range(N):
            if H[i, j] != 0:
                ax.text(j, i, f'{H[i, j]:g}', ha='center', va='center', fontsize=8,
                        color='white' if H[i, j] > 1.5 else 'k')
    ax.set_xticks(range(N)); ax.set_yticks(range(N)); ax.tick_params(labelsize=7)
    ax.set_xlabel('input sample $k$', fontsize=9); ax.set_title(title, fontsize=10)
    ax.grid(False)
axs[0].set_ylabel('output sample $n$', fontsize=9)
plt.tight_layout(); plt.show()
x = np.random.default_rng(1).normal(size=N)
print('max |H x - T{x}| over the five systems:', max(np.max(np.abs(system_matrix(T) @ x - T(x))) for _, T in systems))

<b>Figure 9.4.</b> Five linear systems as $8\times 8$ matrices, built column by column from their responses to $\delta[n-k]$. (a) $y = 3x[n]$ is memoryless: the matrix is diagonal. (b) $y = x[n] + x[n-2]$ (Example 1-2) is causal with memory: a lower band. (c) $y = x[n] + x[n+1]$ (Example 4-1) is anticausal: an upper band. (d) The running sum (Examples 2-1 and 3-1) is causal and time-invariant, but its impulse response $u[n]$ is not absolutely summable: a full lower triangle. (e) $y = 3x[3n]$ (Example 3-2) is linear but time-variant, so its diagonals are not constant. (a)&ndash;(d) are Toeplitz (constant along each diagonal) and (e) is not. The nonlinear system of Example 2-2 has no matrix at all. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; systems as operators and matrices.</b> In Module 8 (Signals as Vectors, Systems as Matrices: the SVD) every linear system acting on
$N$ samples became an $N\times N$ matrix whose $k$-th column is the response to $\delta[n-k]$. The properties of this module are shapes of that matrix:
memoryless means diagonal, causal means lower triangular (no output depends on a later input), and time-invariant means constant along each diagonal
(Toeplitz). A nonlinear system such as Example 2-2 has no matrix at all, which is the clearest test of linearity there is.
</div>

<a name="9.7"></a>
## 9.7 Discrete Convolution

Let's now return to the convolution that we studied in Module 7, so that we can be more precise in our definition of **discrete convolution**.

**DISCRETE CONVOLUTION DEFINITION** 

<div class="alert alert-info" role="alert">
The input-output relationship for an <strong>LTI system</strong> is characterized by a <strong>convolution</strong>:

$$y[n] = x[n] \ast h[n]  = \sum_{m=-\infty}^{\infty} x[m]h[n-m] \tag{54} $$

where $\ast$ denotes (discrete) convolution, the same symbol as in Module 7.
</div>

Note that this is similar to the definition of convolution in the case of a continuum; however, it is simpler because it needs only additions and multiplications.

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication.</b> The convolution sum (54) is Module 7's convolution integral with the integral replaced by a sum and
$\mathrm{d}\tau$ by one sample, so the flip-and-slide picture of Module 7 carries over unchanged (Figure 9.5). Module 7 also showed that the convolution of
two coefficient lists is polynomial multiplication; Example 8 below is the product $(1 + 2z + 5z^2 + 10z^3)(1 + 2z + 3z^2 + 4z^3)$ read off coefficient by coefficient.
</div>

The animation in Figure 9.5 evaluates equation (54) for Example 7 below, one output sample per frame. Watch three things. In the top
panel the 10-sample boxcar $h[n-m]$ (green squares) is the impulse response flipped and slid to position $n$. In the middle panel the products $x[m]h[n-m]$
(orange) are nonzero only where the two overlap. Their sum is the new output sample, which is added to the bottom panel. The output grows while the boxcar is still
moving onto the input ($n < 10$), peaks at $n = 9$, and then decays as the boxcar slides along the shrinking tail of $0.9^m$.

In [ ]:
C = DSP_COLORS
a, L = 0.9, 10                               # Example 7: x[m] = a^m u[m], h = 10-sample boxcar
m = np.arange(-12, 32)
x = np.where(m >= 0, a**np.clip(m, 0, None), 0.0)
h = lambda k: ((k >= 0) & (k < L)).astype(float)
n_frames = 30                                # n = 0 ... 29
y_all = np.array([np.sum(x*h(n - m)) for n in range(n_frames)])
nn = np.arange(-2, 30)
y_exact = np.where(nn < 0, 0, np.where(nn < L, (1 - a**(nn + 1))/(1 - a), a**(nn - 9)*(1 - a**L)/(1 - a)))
fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(10, 7.6), sharex=True)
mx, sx, _ = a1.stem(m, x, basefmt='k-', label='$x[m] = 0.9^m u[m]$')
plt.setp(sx, color=C['input']); plt.setp(mx, color=C['input'], ms=5)
hl, = a1.plot([], [], 's', color=C['impulse_resp'], ms=6, mfc='none', mew=1.8, label='$h[n-m]$ (flipped, shifted)')
a1.set_ylim(-0.15, 1.25); a1.legend(loc='upper right', fontsize=9); a1.set_ylabel('amplitude')
pl = a2.bar(m, np.zeros_like(x), width=0.6, color=C['highlight'])
a2.set_ylim(0, 1.15); a2.set_ylabel('$x[m]\\,h[n-m]$')
a3.plot(nn, y_exact, color=C['continuous'], lw=1, alpha=0.5, label='equation for $y[n]$ (Example 7)')
yl = a3.stem(np.arange(n_frames), np.full(n_frames, np.nan), basefmt='k-')
plt.setp(yl[1], color=C['output']); plt.setp(yl[0], color=C['output'], ms=5)
cur, = a3.plot([], [], 'o', color=C['output'], ms=11, mfc='none', mew=2)
a3.set_ylim(-0.4, 7.6); a3.set_xlim(-12.5, 31.5); a3.set_xlabel('sample index $m$ (top, middle) and $n$ (bottom)')
a3.set_ylabel('$y[n]$'); a3.legend(loc='upper right', fontsize=9)
title = fig.suptitle('', fontsize=13)

def update(k):
    hv = h(k - m)
    hl.set_data(m[hv > 0], hv[hv > 0])
    for rect, v in zip(pl, x*hv):
        rect.set_height(v)
    yv = np.full(n_frames, np.nan); yv[:k + 1] = y_all[:k + 1]
    yl[0].set_ydata(yv)
    yl[1].set_segments([[[i, 0], [i, v]] for i, v in enumerate(yv) if np.isfinite(v)])
    cur.set_data([k], [y_all[k]])
    title.set_text(f'$n$ = {k}:   $y[n] = \\sum_m x[m]\\,h[n-m]$ = {y_all[k]:.4f}')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=400, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 9.5.</b> Discrete convolution of $x[m] = 0.9^m u[m]$ (blue) with the boxcar $h[n] = u[n] - u[n-10]$, for $n = 0, \dots, 29$ (30 frames). Top: the input and the flipped, shifted impulse response $h[n-m]$ (green squares). Middle: the products $x[m]\,h[n-m]$. Bottom: the output $y[n]$ filled in one sample per frame (red) over the analytic result of Example 7 (gray): $y[9] = 6.5132$ and $y[19] = 2.2710$. <i>(Python-generated.)</i>

**COMMUTATIVITY** 

Convolution is **commutative**; that is 

$$ 
\begin{eqnarray}
x[n]\ast h[n] &=& h[n] \ast x[n] \tag{55} \\
\, &=& \sum_{m=-\infty}^{\infty} x[m]h[n-m] = \sum_{m=-\infty}^{\infty} h[n-m]x[m] \tag{56}
\end{eqnarray}
$$

and

$$
\begin{eqnarray}
y[n] &=& x[n]\ast h_1[n] \ast h_2[n]  \\ 
   \,&=& x[n]\ast h_2[n] \ast h_1[n] \tag{57}  \\
   \,&=& x[n]\ast\left(h_2[n] \ast h_1[n] \right) \\
\end{eqnarray}
$$

Figure 9.6 shows the commutative and associative properties of convolution as block diagrams.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
def block_row(ax, y, items, x0=0.3, gap=0.9, w=1.7):
    """Draw input -> boxes -> output along height y; returns the x of the last arrow head."""
    x = x0
    ax.text(x, y, '$x[n]$', ha='right', va='center', fontsize=13)
    for lab in items:
        ax.annotate('', xy=(x + gap, y), xytext=(x + 0.05, y), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
        x += gap
        ax.add_patch(FancyBboxPatch((x, y - 0.32), w, 0.64, boxstyle='round,pad=0.04', fc='white', ec=C['impulse_resp'], lw=2))
        ax.text(x + w/2, y, lab, ha='center', va='center', fontsize=13)
        x += w
    ax.annotate('', xy=(x + gap, y), xytext=(x + 0.05, y), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
    ax.text(x + gap + 0.08, y, '$y[n]$', ha='left', va='center', fontsize=13)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off'); ax.set_xlim(-0.6, 7.6); ax.set_ylim(0.35, 3.65)
block_row(ax, 3.2, ['$h_1[n]$', '$h_2[n]$'])
block_row(ax, 2.0, ['$h_2[n]$', '$h_1[n]$'])
block_row(ax, 0.8, ['$h_1[n] \\ast h_2[n]$'], w=4.3)
for yy in (2.6, 1.4):
    ax.text(3.6, yy, '=', ha='center', va='center', fontsize=18, color=C['neutral'])
plt.tight_layout(); plt.show()
rng = np.random.default_rng(3)
x, h1, h2 = rng.normal(size=20), rng.normal(size=5), rng.normal(size=7)
print('max |(x*h1)*h2 - (x*h2)*h1| =', np.max(np.abs(np.convolve(np.convolve(x, h1), h2) - np.convolve(np.convolve(x, h2), h1))))
print('max |(x*h1)*h2 - x*(h1*h2)| =', np.max(np.abs(np.convolve(np.convolve(x, h1), h2) - np.convolve(x, np.convolve(h1, h2)))))

<b>Figure 9.6.</b> Commutativity and associativity of convolution. Passing $x[n]$ through $h_1[n]$ and then $h_2[n]$ (top) gives the same output as the reverse order (middle) and as a single system with impulse response $h_1[n] \ast h_2[n]$ (bottom). The printed numbers check this for random sequences. <i>(Python-generated.)</i>

**LINEARITY:** 

Because an LTI system is linear, the following holds:

$$y[n] = x[n] \ast \left( h_1[n]+h_2[n] \right) = x[n] \ast h_1[n] + x[n] \ast h_2[n] \tag{58} $$

Figure 9.7 shows this distributive (linearity) property of convolution.

In [ ]:
from matplotlib.patches import FancyBboxPatch
C = DSP_COLORS
def block_row(ax, y, items, x0=0.3, gap=0.9, w=1.7):
    """Draw input -> boxes -> output along height y; returns the x of the last arrow head."""
    x = x0
    ax.text(x, y, '$x[n]$', ha='right', va='center', fontsize=13)
    for lab in items:
        ax.annotate('', xy=(x + gap, y), xytext=(x + 0.05, y), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
        x += gap
        ax.add_patch(FancyBboxPatch((x, y - 0.32), w, 0.64, boxstyle='round,pad=0.04', fc='white', ec=C['impulse_resp'], lw=2))
        ax.text(x + w/2, y, lab, ha='center', va='center', fontsize=13)
        x += w
    ax.annotate('', xy=(x + gap, y), xytext=(x + 0.05, y), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
    ax.text(x + gap + 0.08, y, '$y[n]$', ha='left', va='center', fontsize=13)

fig, ax = plt.subplots(figsize=(7.6, 3.6))
ax.axis('off'); ax.set_xlim(-0.45, 6.75); ax.set_ylim(-0.05, 3.1)
# (top) parallel branches joined by a summing node
ax.text(0.3, 2.2, '$x[n]$', ha='right', va='center', fontsize=13)
ax.plot([0.35, 1.0], [2.2, 2.2], 'k', lw=1.6); ax.plot([1.0, 1.0], [1.65, 2.75], 'k', lw=1.6)
for yy, lab in ((2.75, '$h_1[n]$'), (1.65, '$h_2[n]$')):
    ax.annotate('', xy=(1.8, yy), xytext=(1.0, yy), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
    ax.add_patch(FancyBboxPatch((1.8, yy - 0.26), 1.7, 0.52, boxstyle='round,pad=0.04', fc='white', ec=C['impulse_resp'], lw=2))
    ax.text(2.65, yy, lab, ha='center', va='center', fontsize=13)
    ax.plot([3.55, 4.3, 4.3], [yy, yy, 2.2 + (0.18 if yy > 2.2 else -0.18)], 'k', lw=1.6)
ax.add_patch(plt.Circle((4.3, 2.2), 0.18, fc='white', ec='k', lw=1.6)); ax.text(4.3, 2.2, '+', ha='center', va='center', fontsize=13)
ax.annotate('', xy=(5.3, 2.2), xytext=(4.48, 2.2), arrowprops=dict(arrowstyle='-|>', lw=1.6, color='k'))
ax.text(5.38, 2.2, '$y[n]$', ha='left', va='center', fontsize=13)
ax.text(2.65, 1.12, '=', ha='center', va='center', fontsize=18, color=C['neutral'])
# (bottom) the single equivalent system
block_row(ax, 0.35, ['$h_1[n] + h_2[n]$'], x0=1.2, w=3.0)
plt.tight_layout(); plt.show()
rng = np.random.default_rng(4)
x, h1, h2 = rng.normal(size=20), rng.normal(size=6), rng.normal(size=6)
print('max |x*(h1+h2) - (x*h1 + x*h2)| =', np.max(np.abs(np.convolve(x, h1 + h2) - (np.convolve(x, h1) + np.convolve(x, h2)))))

<b>Figure 9.7.</b> The distributive property of convolution. Two systems in parallel whose outputs are added (top) are equivalent to one system with impulse response $h_1[n] + h_2[n]$ (bottom). The printed number checks this for random sequences. <i>(Python-generated.)</i>

### EXAMPLE 6

**Q:** Compute the output $y[n]$ if the input is $x[n]=u[n]$ and the LTI system impulse response is $h[n]=\delta[n]+\frac{1}{2}\delta[n-1]$.  Determine the stability, causality and memory of the system.

**A:** The discrete convolution equation is given by:

$$
\begin{eqnarray}
y[n] &=& x[n] \ast h[n]=\sum_{m=-\infty}^{\infty} x[m]h[n-m] \tag{59}\\
   \,&=& \sum_{m=-\infty}^{\infty} u[m] \left(\delta[n-m]+\frac{1}{2}\delta[n-1-m]\right) \tag{60}  \\
 \,&=& \sum_{m=0}^{\infty}\left(\delta[n-m]+\frac{1}{2}\delta[n-1-m]\right)  \tag{61} \\  
  \,&=& \sum_{m=0}^{\infty}\delta[n-m]+\frac{1}{2}\sum_{m=0}^{\infty}\delta[n-1-m] \tag{62}\\
   \,&=& u[n]+\frac{1}{2}u[n-1] \tag{63}
\end{eqnarray}
$$

Since $\sum_{n=-\infty}^{\infty}|h[n]| = \sum_{n=0}^{1}|h[n]|=\frac{3}{2} \lt \infty$ and $h[n]=0$ for $n\lt0$ the system is stable, causal and has memory.

In [ ]:
C = DSP_COLORS

n = np.arange(-25, 26)
x = (n >= 0).astype(float)
h = 1.0*(n == 0) + 0.5*(n == 1)
y1 = 1.0*(n >= 0) + 0.5*(n >= 1)                                # analytic result
y2 = np.convolve(x, h, mode='full')[25:76]                                # numerical result (np.convolve)
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for ax, (nn_, v, col, ttl) in zip(axs.flat, [(n, x, C['input'], '(a) $x[n] = u[n]$, unit step'), (n, h, C['impulse_resp'], '(b) $h[n] = \\delta[n] + \\frac{1}{2}\\delta[n-1]$'),
                                            (n, y1, C['output'], '(c) $y[n]$, analytic'), (np.arange(len(y2)) + n[0], y2, C['output'], '(d) $y[n]$, numerical (np.convolve)')]):
    ml, sl, bl = ax.stem(nn_, v, basefmt='k-')
    plt.setp(sl, color=col, lw=1.3); plt.setp(ml, color=col, ms=5)
    ax.set_xlim(-25.5, 25.5); ax.set_ylim(-0.3, 1.8)
    ax.set_xlabel('time sample $n$'); ax.set_ylabel('amplitude'); ax.set_title(ttl, fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 9.8.</b> Example 6. (a) The unit step input $u[n]$; (b) the impulse response $h[n] = \delta[n] + \frac{1}{2}\delta[n-1]$; (c) the analytic output, equation (63); (d) the numerical output from <code>np.convolve</code>. (c) and (d) agree exactly. <i>(Python-generated.)</i>

Let's now calculate the difference between the analytic and numerical results:

In [ ]:
print('Difference between numerical and analytic results (all 51 samples):')
print(y2 - y1)

### EXAMPLE 7

**Q:** Compute the output $y[n]$ if the input is $x[n]=a^n u[n]$ and the LTI system impulse response is $h[n]=u[n]-u[n-10]$.  Determine the stability and causality of the system.

**A:** Using the above we have:

$$
\begin{eqnarray}
y[n] & = & \sum_{m=-\infty}^{\infty} x[m] h[n-m] \tag{64}\\
\  \,& = & \sum_{m=-\infty}^{\infty} a^m u[m] \left(u[n-m]-u[n-m-10]\right) \tag{65}\\
\  \,& = & \sum_{m=0}^{\infty} a^m \left(u[n-m]-u[n-m-10]\right) \tag{66}\\
\  \,& = & \sum_{m=0}^{\infty} a^m u[n-m]- \sum_{m=0}^{\infty} a^m  u[n-m-10] \tag{67}\\
\end{eqnarray}
$$

Define $y_1[n]=\sum_{m=0}^{\infty} a^m u[n-m]$ and $y_2[n]=\sum_{m=0}^{\infty} a^m  u[n-m-10]$ such that $y[n]=y_1[n]-y_2[n]$.  By employing a change of variable (i.e., $n-m=k$ or $m=n-k$)

$$
\begin{eqnarray}
y_1[n] &=& \sum_{m=0}^{\infty} a^m u[n-m] = \sum^{n}_{k=-\infty} a^{n-k} u[k], \quad k=n-m \tag{68}
\end{eqnarray}
$$

Since $u[k]=0$ for $k<0$, $y_1[n]=0$ for $n<0$.  For $n\ge 0$, $y_1[n]$ is given by a [geometric series](http://mathworld.wolfram.com/GeometricSeries.html)

$$ y_1[n] =\sum_{k=0}^{n} a^{n-k} u[k]=\sum_{k=0}^{n} a^{n-k} = 1+a+...+a^n = \frac{1-a^{n+1}}{1-a},n\ge0 \tag{69} $$

That is, 

$$ y_1[n] = \frac{1-a^{n+1}}{1-a} u[n] \tag{70} $$

Similarly, $y_2[n]$ is:

$$ 
\begin{eqnarray}
y_2[n] &=& \sum_{m=0}^{\infty} a^m u[n-10-m]  \\
     \,&=& \sum_{k=-\infty}^{n-10}a^{n-10-k} u[k], \quad k=n-10-m \tag{71}
\end{eqnarray}
$$

Since $u[k]=0$ for $k<0$, $y_2[n]=0$ for $n<10$.  Thus, for $n \ge 10$, $y_2[n]$ is:

$$ y_2[n] = \sum_{k=0}^{n-10}a^{n-10-k} u[k] = \sum_{k=0}^{n-10}a^{n-10-k}  =1+a+...+a^{n-10} = \frac{1-a^{n-9}}{1-a},n \ge 10 \tag{72} $$

That is,

$$ y_2[n] = \frac{1-a^{n-9}}{1-a}u[n-10] \tag{73} $$

Combining the results we have

$$ y[n] = \frac{1-a^{n+1}}{1-a} u[n] - \frac{1-a^{n-9}}{1-a} u[n-10] \tag{74} $$

or 

$$y[n] = \left\{
\begin{array}{ll}
0, & n<0\\
\frac{1-a^{n+1}}{1-a}, & 0\le n <10 \\
\frac{a^{n-9}(1-a^{10})}{1-a}, & n \ge 10
\end{array}
\right.{}
\tag{75}
$$

Note: because $h[n]=0$ for $n<0$ the system is causal. However, is it stable? And is the output bounded (think about the value of $a$ above)?

<div class="alert alert-info">
<b>Synthesis &mdash; prerequisite bridge: the geometric series.</b> Equation (69) is the MATH 213 finite geometric sum $1 + a + \dots + a^n = (1 - a^{n+1})/(1 - a)$.
The same series decides stability: $\sum_{n\ge 0}|a^n|$ converges only if $|a| < 1$ (the ratio test), so an impulse response $h[n] = a^n u[n]$ is stable exactly when
$|a| < 1$. Example 7's 10-sample boxcar $h[n]$, by contrast, is stable for any $a$; the value of $a$ only decides whether the <i>input</i> is bounded.
</div>

In [ ]:
C = DSP_COLORS
a = 0.9
n = np.arange(0, 40)
x = a**n
h = ((n >= 0) & (n < 10)).astype(float)
y1 = np.where(n < 10, (1 - a**(n + 1))/(1 - a), a**(n - 9)*(1 - a**10)/(1 - a))                                # analytic result
y2 = np.convolve(x, h, mode='full')[:40]                                # numerical result (np.convolve)
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for ax, (nn_, v, col, ttl) in zip(axs.flat, [(n, x, C['input'], '(a) $x[n] = 0.9^n u[n]$'), (n, h, C['impulse_resp'], '(b) $h[n] = u[n] - u[n-10]$'),
                                            (n, y1, C['output'], '(c) $y[n]$, analytic'), (np.arange(len(y2)) + n[0], y2, C['output'], '(d) $y[n]$, numerical (np.convolve)')]):
    ml, sl, bl = ax.stem(nn_, v, basefmt='k-')
    plt.setp(sl, color=col, lw=1.3); plt.setp(ml, color=col, ms=5)
    ax.set_xlim(-1.5, 40.5); ax.set_ylim(-0.4, 7.5)
    ax.set_xlabel('time sample $n$'); ax.set_ylabel('amplitude'); ax.set_title(ttl, fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 9.9.</b> Example 7 with $a = 0.9$. (a) The input $x[n] = 0.9^n u[n]$; (b) the impulse response $h[n] = u[n] - u[n-10]$; (c) the analytic output, equation (75); (d) the numerical output from <code>np.convolve</code>, which matches (c) to round-off. <i>(Python-generated.)</i>

Let's now calculate the difference between the analytic and numerical results:

In [ ]:
print('Maximum difference between numerical and analytic outputs (40 samples):', np.max(np.abs(y2 - y1)))

### EXAMPLE 8

**Q**: Determine $y[n]=x[n]\ast h[n]$ where $x[n]$ and $h[n]$ are given by:

$$ x[n] = \left\{\begin{array}{ll} n^2+1, & 0\le n\le 3 \\ 0, & \mathrm{otherwise}\end{array}\right.{} \tag{76} $$

and

$$ h[n] = \left\{\begin{array}{ll} n+1, & 0\le n\le 3 \\ 0, & \mathrm{otherwise}\end{array}\right.{} \tag{77} $$

**A**: Here, the lengths of both $x[n]$ and $h[n]$ are finite.  More precisely, $x[0]=1$, $x[1]=2$, $x[2]=5$, and $x[3]=10$ and $h[0]=1$, $h[1]=2$, $h[2]=3$, and $h[3]=4$; while all others have zero values.  We write the following:

$$
\begin{eqnarray}
y[n]& =& \sum_{m=-\infty}^{\infty} x[m] h[n-m]\tag{78} \\ 
\,  & = &\sum_{m=0}^{3} x[m] h[n-m] \tag{79}\\
\,  & =& x[0]h[n]+x[1]h[n-1]+x[2]h[n-2]+x[3]h[n-3] \tag{80} \\
\,  & =& 1\cdot h[n]+2\cdot h[n-1]+5\cdot h[n-2]+10\cdot h[n-3] \tag{81} \\
\end{eqnarray}
$$

where equation (78) is the definition of discrete convolution, equation (79) has the range restricted because $x[n]$ is zero for values of $m$ outside of $0\le m\le 3$, equation (80) is expanding out the sum in equation (79), and equation (81) is replacing the actual non-zero values of $x[n]$ in equation (80).

By considering the **non-zero** values of $h[n]$ we obtain:
$$
y[n]=
\left\{
\begin{array}{ll}
1, & \quad n=0 \\
4, & \quad n=1 \\
12, & \quad n=2 \\
30, & \quad n=3 \tag{82}\\
43, & \quad n=4 \\
50, & \quad n=5 \\
40, & \quad n=6 \\
0 , & \quad \mathrm{otherwise} \\
\end{array}
\right.{}
$$

In [ ]:
C = DSP_COLORS

n = np.arange(0, 7)
x = np.where(n <= 3, n**2 + 1.0, 0.0)
h = np.where(n <= 3, n + 1.0, 0.0)
y1 = np.array([1, 4, 12, 30, 43, 50, 40], dtype=float)                                # analytic result
y2 = np.convolve(x[:4], h[:4], mode='full')                                # numerical result (np.convolve)
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for ax, (nn_, v, col, ttl) in zip(axs.flat, [(n, x, C['input'], '(a) $x[n]$'), (n, h, C['impulse_resp'], '(b) $h[n]$'),
                                            (n, y1, C['output'], '(c) $y[n]$, analytic (by hand)'), (np.arange(len(y2)) + n[0], y2, C['output'], '(d) $y[n]$, numerical (np.convolve)')]):
    ml, sl, bl = ax.stem(nn_, v, basefmt='k-')
    plt.setp(sl, color=col, lw=1.3); plt.setp(ml, color=col, ms=5)
    ax.set_xlim(-0.8, 6.8); ax.set_ylim(0, 58)
    ax.set_xlabel('time sample $n$'); ax.set_ylabel('amplitude'); ax.set_title(ttl, fontsize=12)
plt.tight_layout(); plt.show()

<b>Figure 9.10.</b> Example 8. (a) The input $x[n]$ and (b) the impulse response $h[n]$ of equations (76) and (77); (c) the output computed by hand, equation (82); (d) the numerical output from <code>np.convolve</code>. <i>(Python-generated.)</i>

Let's now calculate the difference between the analytic and numerical results:

In [ ]:
print('Difference between numerical and analytic outputs (all 7 samples):')
print(y2 - y1)

<a name="9.8"></a>
## 9.8 Linear Constant-Coefficient Difference Equations

Another important class of equations that can be developed are linear constant coefficient [difference equation](https://en.wikipedia.org/wiki/Linear_difference_equation) or LCCDE for short. Mathematically, these can be defined by the following systems:

<div class="alert alert-info" role="alert">
For an LTI system its input $x[n]$ and output $y[n]$ are related via a $N$th-order linear constant-coefficient difference equation:

$$ \sum_{k=0}^{N} a_k y[n-k] = \sum_{k=0}^{M} b_k x[n-k] \tag{83} $$

which is useful to check whether (or not) a system is both linear and time-invariant. 
</div>

LCCDEs commonly arise in systems exhibiting discrete (rather than continuous) behavior.  Examples include:
* Compound interest that is paid on a discrete interval (e.g., monthly).
* Population growth (the number of a species born has to be an integer!)

A system with coefficient $a_k\neq 0, k\ge 1$ is said to have **feedback** since the current output value depends on the *previous* output values.  A filter exhibiting such a characteristic is called a **recursive filter** or an **infinite-impulse response (IIR) filter**.  For an LTI system where $a_k=0, k\ge 1$ does not have feedback, and is called a **nonrecursive filter** or a **finite-impulse response (FIR) filter**.  Module 15 designs both kinds.

### EXAMPLE 9 

Determine if the following input-output relationships correspond to LTI systems:

(1) $\quad y[n]=0.1y[n-1]+x[n]+x[n-1]$:  
   * Yes, this is an LTI system with $N=M=1$ and $a_0=1, a_1=-0.1$ and $b_0=b_1=1$.


(2) $\quad y[n]=x[n+1]+x[n]$
   * Yes, we note that $y[n]=x[n+1]+x[n] \rightarrow y[n-1]=x[n]+x[n-1]$, which is an LTI system of the form of equation (83) with $N=0$, $M=1$, $a_0=1$ and $b_0=b_1=1$ (after the index shift; the original form uses the future input $x[n+1]$, so it is non-causal).


(3) $\quad y[n] = 1/x[n] $
   * No, because $x[n]$ and $y[n]$ are not linear in this system.

### EXAMPLE 10 

**Q:** You invest money in a bank account with input of $x[n]=x_0\delta[n]$. You receive an annual interest rate of $p$%, and conversion period equal to $\alpha$ (i.e., monthly) so that the interest earned for the period is equal to $\alpha p$% $\left(\mathrm{or}\,\frac{\alpha p}{100}\right)$ of the amount on deposit at the start of the period. What is your balance after 20 years?

**A:** You can model this system as the following:

$$
\left\{
\begin{array}{c}
amount\,on \\
deposit\\
after\,n\\
conversion \\
periods
\end{array}
\right\}
=
\left\{
\begin{array}{c}
amount\,on \\
deposit\\
after\,n-1\\
conversion \\
periods
\end{array}
\right\}
+
\frac{\alpha p}{100}
\left\{
\begin{array}{c}
amount\,on \\
deposit\\
after\,n-1\\
conversion \\
periods
\end{array}
\right\}
+
\left\{
\begin{array}{c}
amount\, \\
deposited \\
during\\
n-1 \\
period
\end{array}
\right\}
\tag{84}
$$

We may write:

$$ y[n] = y[n-1]+\frac{\alpha p}{100}y[n-1] + x[n-1] = \left(1+\frac{\alpha p}{100}\right)\,y[n-1] + x[n-1] \tag{85} $$

where $x[n]=x_0\delta[n]$ is the *input* time series.  The deposit enters at $n=1$ ($y[1] = x_0$). Recognizing that this system has *feedback*, we find the following closed form of the recursion, with $r = \alpha p/100$:

$$y[n]= \left(1+\frac{\alpha p}{100}\right)^{n-1} y[1] = \left(1+\frac{\alpha p}{100}\right)^{n-1} x_0 = \left(1+r\right)^{n-1} x_0, \quad n \ge 1 \tag{86} $$

Note that this exponential growth is an example of a IIR filter! 

In [ ]:
C = DSP_COLORS
alpha, p, x0 = 1/12, 7.0, 100000.0           # monthly conversion, 7 percent per year, initial deposit
r = alpha*p/100
nmax = 240                                   # 20 years of monthly periods
y = np.zeros(nmax + 1)
y[1] = x0                                    # x[n-1] = x0*delta[n-1]: the deposit enters at n = 1 (equation 50)
for n in range(2, nmax + 1):
    y[n] = (1 + r)*y[n - 1]                  # recursion (feedback)
n = np.arange(nmax + 1)
closed = np.where(n >= 1, x0*(1 + r)**(n - 1), 0.0)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.6))
axa.plot(n*alpha, y, '.', color=C['output'], ms=4)
axa.set_xlabel('time (years)'); axa.set_ylabel('balance (\\$)'); axa.set_title('(a) Exponential growth of the balance')
rel = np.abs(y[1:] - closed[1:])/closed[1:]
eps = np.finfo(float).eps
axb.plot(n[1:]*alpha, rel/eps, color=C['time_domain'])
axb.set_xlabel('time (years)'); axb.set_ylabel(r'relative difference / machine $\epsilon$'); axb.set_title('(b) Recursion vs closed form (round-off)')
plt.tight_layout(); plt.show()
print(f'Balance after 20 years: ${y[nmax]:,.2f}')

<b>Figure 9.11.</b> Example 10: a deposit of \$100,000 at 7% per year, compounded monthly. (a) The balance grows exponentially to \$401,532 after 20 years (240 periods). (b) The relative difference between the recursion (equation 85) and the closed form (equation 86) stays at the level of round-off: the two are the same filter. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; prerequisite bridge: difference and differential equations.</b> Example 10's $y[n] = (1 + r)\,y[n-1]$ is the discrete twin of the MATH 225 growth
equation $\mathrm{d}y/\mathrm{d}t = ky$, whose solution $e^{kt}$ becomes $(1 + r)^n$. The PHGN 200 RC circuit $\tau\,\mathrm{d}y/\mathrm{d}t + y = x$, stepped with
Euler's method, becomes an LCCDE with feedback (Microquestion 9.2). An IIR filter is a discretized ODE, and the ODE's stability does not guarantee the discretization's.
</div>

### EXAMPLE 11 

You have borrowed some money from the bank for a mortgage.  Assuming a starting loan of $y[0]=x_0$, an annual interest rate of $p$%, and conversion period equal to $\alpha$ (e.g., monthly), the interest earned for the period is equal to $\alpha p$% (or $\frac{\alpha p}{100}$) of the amount on deposit at the start of the period.  You have a choice of paying either a constant $\beta$ per $\alpha$ period (i.e., $x[n-1]=\beta$) or paying an exponentially decaying amount per month (i.e., $x[n-1]=R_0\mathrm{e}^{-\gamma n}$).  What do you choose?

$$
\left\{
\begin{array}{c}
amount\,on \\
mortgage\\
after\,n\\
conversion \\
periods
\end{array}
\right\}
=
\left\{
\begin{array}{c}
amount\,on \\
mortgage\\
after\,n-1\\
conversion \\
periods
\end{array}
\right\}
+
\frac{\alpha p}{100}
\left\{
\begin{array}{c}
amount\,on \\
mortgage\\
after\,n-1\\
conversion \\
periods
\end{array}
\right\}
-
\left\{
\begin{array}{c}
amount \\
paid\\
during\\
the\,n-1 \\
period
\end{array}
\right\}
\tag{87}
$$

This is close to the example above, except that now the system has input $x[n-1]$.  

$$ y[n] = y[n-1]+\frac{\alpha p}{100}y[n-1] - x[n-1] = \left(1+\frac{\alpha p}{100}\right)\,y[n-1]-x[n-1] \tag{88}$$

where $x[n-1]$ is either $x[n-1]=\beta$ or $x[n-1]=R_0\mathrm{e}^{-\gamma n}$.  Now it's much harder to determine an analytic solution for this system (though not impossible)! Let's see what the difference in the different payment schemes actually is.

In [ ]:
C = DSP_COLORS
nlen = 12*20                                 # loan period: 12 months times 20 years
alpha, p, x0 = 1/12, 5.0, 100000.0           # monthly payments, 5 percent interest, starting balance
pay = 0.0066*x0                              # constant payment
opr = 1 + alpha*p/100
n = np.arange(nlen)
payments = 1.55*pay*np.exp(-0.005*n)         # exponentially decaying payments
zero, cons, expo = np.zeros(nlen), np.zeros(nlen), np.zeros(nlen)
zero[0], cons[0], expo[0] = x0, x0 - pay, x0 - payments[0]
for k in range(1, nlen):
    zero[k] = opr*zero[k - 1]
    cons[k] = opr*cons[k - 1] - pay
    expo[k] = opr*expo[k - 1] - payments[k - 1]
fig, (axa, axb) = plt.subplots(1, 2, figsize=(13, 4.8))
axa.plot(n*alpha, 0*n, color=C['time_domain'], label='no payment')
axa.plot(n*alpha, pay*np.ones(nlen), color=C['freq_domain'], label='constant payment')
axa.plot(n*alpha, payments, color=C['impulse_resp'], label='exponential payment')
axa.set_xlim(0, 20); axa.set_ylim(-50, 1100)
axa.set_xlabel('time (years)'); axa.set_ylabel('payment each month (\\$)'); axa.set_title('(a) Payment schedules'); axa.legend(fontsize=10)
axb.plot(n*alpha, zero, color=C['time_domain'], label='no payments')
axb.plot(n*alpha, cons, color=C['freq_domain'], label='constant payments')
axb.plot(n*alpha, expo, color=C['impulse_resp'], label='exponential payments')
axb.set_xlim(0, 20); axb.set_ylim(0, 300000)
axb.set_xlabel('time (years)'); axb.set_ylabel('balance (\\$)'); axb.set_title('(b) Is paying off your debts worth it?'); axb.legend(fontsize=10)
plt.tight_layout(); plt.show()
print(f'TOTAL CONSTANT PAYMENTS   : ${nlen*pay:,.2f}')
print(f'TOTAL EXPONENTIAL PAYMENTS: ${payments.sum():,.2f}')
print("You'd better do constant payments!" if nlen*pay < payments.sum() else "You'd better do exponential payments!")

<b>Figure 9.12.</b> Example 11: a \$100,000 mortgage at 5% per year with monthly payments. (a) Three payment strategies: no payment (blue), a constant payment (red), and an exponentially decaying payment (green). (b) The resulting balances. The printed totals compare what each strategy pays in total. It is best to pay more at the start of the mortgage than at the end! <i>(Python-generated.)</i>

### EXAMPLE 12 

**Q:** Moneyball!  Your baseball manager wants you to design a filter to automatically calculate the running average of the hitting percentage of each player for the previous $M$ at bats.  

**A:** You note that the requested analysis is *non-recursive* (i.e., no feedback) since the output average after the next at bat does not depend on the output average of the previous. Assuming that the input hits are given by $x[n]$, you write the following averaging formula:

$$ y[n]=\sum_{m=0}^{M-1} \frac{x[n-m]}{M} \tag{89} $$

Note that the averaging process as written is an example of a FIR filter.

In [ ]:
C = DSP_COLORS
N, p = 2000, 0.3                             # career at bats; a ".300 hitter"
rng = np.random.default_rng(404)             # seeded so the figure is reproducible
values = rng.binomial(1, p, N).astype(float) # 1 = hit, 0 = out
fig, axs = plt.subplots(2, 2, figsize=(13, 6.4), sharex=True, sharey=True)
for ax, M, lab in zip(axs.flat, (10, 50, 200, 500), 'abcd'):
    yM = np.convolve(values, np.ones(M)/M)[:N]   # equation 54: y[n] = (1/M) sum_{m=0}^{M-1} x[n-m]
    ax.plot(values, '.', color=C['input'], ms=2, alpha=0.4)
    ax.plot(yM, color=C['output'], lw=1.5)
    ax.axhline(p, ls='--', color=C['neutral'], lw=1)
    ax.set_title(f'({lab}) running average, M = {M}', fontsize=12); ax.set_ylim(-0.05, 1.05)
for ax in axs[1]:
    ax.set_xlabel('at bat $n$')
for ax in axs[:, 0]:
    ax.set_ylabel('hit (dots), average (line)')
plt.tight_layout(); plt.show()

<b>Figure 9.13.</b> Example 12: running averages of a simulated .300 hitter (2000 at bats; dots are hits and outs) computed with equation (89) for (a) $M = 10$, (b) $M = 50$, (c) $M = 200$ and (d) $M = 500$ at bats. Longer averages are smoother but respond more slowly; the dashed line is the true rate 0.3. <i>(Python-generated.)</i>

You then notice that you can write this in a different way:

$$ y[n] = y[n-1] +\frac{1}{M}\left(x[n] - x[n-M]\right) \tag{90}$$

Which way is "better"? How would you test this?

<a name="9.9"></a>
## 9.9 Summary

Discrete signals are sequences of numbers built from unit impulses, and real records add random noise to a deterministic part.
Five properties (memory, linearity, time invariance, causality and stability) classify discrete systems, and for linear systems they can be read off the shape of
the system matrix. An LTI system is completely described by its impulse response, and its output is the convolution sum (54). Difference equations with
feedback describe recursive (IIR) filters; without feedback they describe FIR filters such as the running average.

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 9.1 &mdash; A first difference as a matrix</b>

The first-difference system $y[n] = x[n] - x[n-1]$ acts on 4-sample signals ($n = 0, \dots, 3$, with $x[-1] = 0$). Use the input $x = (1, 2, 0, -1)$ and the unit step $u = (1, 1, 1, 1)$.

<b>(a)</b> Build the $4\times 4$ matrix $D$ column by column from the responses to $\delta[n-k]$, and compute $Dx$ and $Du$. Which earlier equation in this module does $Du$ reproduce?

<b>(b)</b> The running sum $S$, with $y[n] = \sum_{k\le n} x[k]$ (Example 2-1), is lower triangular with ones. Show that $SD = I$, the discrete form of the MATH 213 fundamental theorem of calculus (the difference quotient $(x[n] - x[n-1])/T$ approximates $\mathrm{d}x/\mathrm{d}t$). Classify $D$ and $S$ for memory, causality, time invariance and stability (equation 49).

<b>(c)</b> A student says: "Any lower-triangular matrix is a causal LTI system." Give a counterexample and say which property fails.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 9.1</b></summary>

<b>(a)</b> $D = \begin{pmatrix} 1&0&0&0\\ -1&1&0&0\\ 0&-1&1&0\\ 0&0&-1&1 \end{pmatrix}$. $Dx = (1, 1, -2, -1)$ and $Du = (1, 0, 0, 0) = \delta[n]$, which is equation (5): $\delta[n] = u[n] - u[n-1]$.

<b>(b)</b> Write $D = I - Z$, where $Z$ shifts down by one sample. $SZ$ is $S$ with its main diagonal removed (the strictly lower ones), so $SD = S - SZ = I$: summing undoes differencing (a telescoping sum). Both have memory, both are causal (lower triangular) and both are time-invariant (Toeplitz). $D$ is stable ($h = \delta[n] - \delta[n-1]$, $\sum|h| = 2$); $S$ is <b>not</b> ($h = u[n]$, $\sum|h| = \infty$). Even so, $S$ is the exact inverse of $D$ on finite signals: inverting a benign operator can give an unstable one, as Module 8 warned.

<b>(c)</b> $y[n] = n\,x[n-1]$ is lower triangular, so it is causal, and it is linear, but the entries $1, 2, 3$ down its subdiagonal are not constant: it is <b>time-variant</b>, like Example 3-2 and panel (e) of Figure 9.4.
</details>

<div class="alert alert-warning">
<b>Microquestion 9.2 &mdash; An RC circuit becomes a recursive filter</b>

The PHGN 200 RC circuit $\tau\,\mathrm{d}y/\mathrm{d}t + y = x$ with $\tau = RC = 1$ ms is sampled every $T = 0.1$ ms. The backward-Euler difference (MATH 225) gives $y[n] = a\,y[n-1] + (1 - a)\,x[n]$ with $a = \tau/(\tau + T)$. The input is a unit step that starts at $n = 0$.

<b>(a)</b> Identify $N$, $M$, the $a_k$ and the $b_k$ of equation (83). Is the filter FIR or IIR?

<b>(b)</b> Find $h[n]$, check stability with equation (49), and compute the step response $s[n] = 1 - a^{n+1}$ at $n = 20$ ($t = 2$ ms). Compare it with the exact circuit value $1 - e^{-2}$.

<b>(c)</b> A colleague uses forward Euler, $y[n] = (1 - T/\tau)\,y[n-1] + (T/\tau)\,x[n-1]$, with a coarse $T = 2.5$ ms. She argues: "The RC circuit is stable, so my filter is too." Is she right?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 9.2</b></summary>

<b>(a)</b> $N = 1$, $M = 0$; $a_0 = 1$, $a_1 = -a = -0.90909$, $b_0 = 1 - a = 0.09091$. It has feedback ($a_1 \neq 0$), so it is a recursive <b>IIR</b> filter.

<b>(b)</b> $h[n] = (1 - a)\,a^n u[n]$, and $\sum|h| = (1 - a)/(1 - a) = 1 < \infty$ (a geometric series), so the filter is stable. $s[20] = 1 - (10/11)^{21} = 0.8649$, against $1 - e^{-2} = 0.8647$: a 0.02% error.

<b>(c)</b> No. Here $1 - T/\tau = -1.5$, so $h[n] = (T/\tau)(-1.5)^{n-1}u[n-1]$ alternates in sign and grows ($|h[10]| = 2.5 \times 1.5^9 = 96.1$); $\sum|h|$ diverges and the filter is unstable even though the circuit is stable. Backward Euler has $0 < a < 1$ for any $T$ and stays stable. Stability is a property of the <i>discrete</i> system (equation 49); forward Euler needs $T < 2\tau$. The same circuit appears as a continuous convolution in Module 7.
</details>

<div class="alert alert-warning">
<b>Microquestion 9.3 &mdash; A discrete convolutional seismogram</b>

A discrete reflectivity series $r[n] = (1, 0, -0.5, 0, 0.25)$ at $n = 0, \dots, 4$ is convolved with a zero-phase, Ricker-like wavelet $w[n] = (-0.5, 1, -0.5)$ at $n = -1, 0, 1$, as in Module 7\'s convolutional model of a seismic trace, but sample by sample.

<b>(a)</b> Over what range of $n$ is $y = r \ast w$ nonzero, and how many samples does it have? Compute $y[-1]$, $y[0]$ and $y[1]$ from equation (54).

<b>(b)</b> Show that $\sum_n y[n] = \left(\sum r\right)\left(\sum w\right)$, using the polynomial (MATH 213 Cauchy product) view of Section 9.7, and evaluate both sides. What does $\sum w = 0$ say about the trace?

<b>(c)</b> "Because $w[-1] \neq 0$, this wavelet is non-causal, so this model cannot describe a real recording." Evaluate this claim using time invariance.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 9.3</b></summary>

<b>(a)</b> The support is $n = 0 + (-1), \dots, 4 + 1 = -1, \dots, 5$, i.e., $5 + 3 - 1 = 7$ samples. $y[-1] = r[0]w[-1] = -0.5$; $y[0] = r[0]w[0] + r[1]w[-1] = 1$; $y[1] = r[0]w[1] + r[1]w[0] + r[2]w[-1] = -0.5 + 0 + 0.25 = -0.25$. The full output is $(-0.5, 1, -0.25, -0.5, 0.125, 0.25, -0.125)$.

<b>(b)</b> Write $R(z) = \sum r[n] z^n$ and $W(z)$ likewise; then $Y(z) = R(z)W(z)$ (the coefficients convolve), and setting $z = 1$ gives $\sum y = R(1)W(1) = 0.75 \times 0 = 0$, as the full output confirms. A wavelet with no DC component produces a trace with zero mean, whatever the geology. (Real seismic sources radiate no DC either.)

<b>(c)</b> Delaying the wavelet by one sample, $w_c[n] = w[n-1]$, makes it causal, and by time invariance it only delays the trace by one sample. Recorded wavelets are causal; zero-phase wavelets are a processing convention (for example after deconvolution) used with a known time shift. Non-causality here is a matter of where $t = 0$ is put, not of physics.
</details>